In [1]:
import pandas as pd
df = pd.read_csv('C:/Users/chalani/Desktop/smartcare-ai-coursework/data/raw/smartcare_ai_dataset_1000.csv')

In [2]:
# Flag the admitted=1 but room_type missing
df['room_type_missing_flag'] = df['room_type'].isnull() & (df['admitted'] == 1)

In [3]:
# Fill the missing room_type values (906) identified in data understanding section
df['room_type'] = df['room_type'].fillna('Not Recorded')

In [4]:
# Confirm no missing values remain
df.isnull().sum().sum()

np.int64(0)

In [5]:
# Re-check duplicates
df.duplicated().sum() 

np.int64(0)

In [6]:
# Look at the low BMI rows (check with age)
df[df['bmi'] < 15][['age', 'bmi']]

,age,bmi
45,50,14.0
90,40,14.7
141,36,14.4
154,68,14.5
245,57,14.0
293,18,14.5
322,40,14.0
333,6,14.0
564,62,14.0


In [7]:
# Flag (without delete) the BMI outlier
df['bmi_outlier_flag'] = df['bmi'] < 15

In [8]:
# Admitted patients should generally have some room charge - check for admitted=1 but room_charge=0
df[(df['admitted'] == 1) & (df['room_charge_lkr'] == 0)][['admitted', 'room_charge_lkr']]

,admitted,room_charge_lkr
19,1,0
21,1,0
29,1,0
30,1,0
32,1,0
...,...,...
971,1,0
974,1,0
977,1,0
983,1,0


In [9]:
# Flag the admitted=1 but room_charge=0
df['room_charge_missing_flag'] = (df['admitted'] == 1) & (df['room_charge_lkr'] == 0)

In [13]:
# Checks whether the "missing room type" group and the "zero room charge" group are the same 236 patients or different ones
(df['room_type_missing_flag'] & df['room_charge_missing_flag']).sum()

np.int64(236)

In [10]:
# Admitted patients should have length_of_stay > 0
df[(df['admitted'] == 1) & (df['length_of_stay_days'] == 0)]

,record_id,patient_id,age,gender,blood_group,department,diagnosis,appointment_date,waiting_days,previous_appointments,...,medicine_charge_lkr,total_bill_lkr,payment_status,payment_method,no_show,readmitted_30_days,disease_risk_level,room_type_missing_flag,bmi_outlier_flag,room_charge_missing_flag


In [11]:
# Non-admitted patients shouldn't have a room charge at all
df[(df['admitted'] == 0) & (df['room_charge_lkr'] > 0)]

,record_id,patient_id,age,gender,blood_group,department,diagnosis,appointment_date,waiting_days,previous_appointments,...,medicine_charge_lkr,total_bill_lkr,payment_status,payment_method,no_show,readmitted_30_days,disease_risk_level,room_type_missing_flag,bmi_outlier_flag,room_charge_missing_flag


## Data Cleaning Workflow

1. Missing values: Only room_type had missing values (906 of 1000). 670 were 
   patients not admitted (expected — filled as "Not Recorded"). The remaining 
   236 were admitted patients missing room_type, flagged with 
   room_type_missing_flag before filling, so the information isn't lost even 
   though the column itself is now filled.

2. Duplicates: No duplicate rows found (df.duplicated().sum() = 0).

3. Outlier identification: 9 rows had bmi below 15. Checked against age — 8 of 
   9 were adults, so this is a genuine clinical outlier, not explained by young 
   patients. Flagged with bmi_outlier_flag rather than removed, since deleting 
   rows without strong justification risks losing valid data.

4. Cross-column consistency check: found that the same 236 admitted patients 
   missing room_type also had zero room_charge_lkr (confirmed by 
   (df['room_type_missing_flag'] & df['room_charge_missing_flag']).sum() = 236, 
   a full match). This means it's one single data entry gap for this subgroup, 
   not two separate issues. Flagged with room_charge_missing_flag rather than 
   removed, since dropping 236 rows would mean losing 23.6% of all admitted 
   patients. No issues found with length_of_stay_days or non-admitted billing 
   consistency.

In [18]:
# Create a blood pressure category
df['bp_category'] = pd.cut(df['systolic_bp'], bins=[0,120,140,200], labels=['Normal','Elevated','High'])

In [19]:
# Create a BMI category
df['bmi_category'] = pd.cut(df['bmi'], bins=[0,18.5,25,30,50], labels=['Underweight','Normal','Overweight','Obese'])

In [20]:
# Create a missed appointment rate
df['missed_appointment_rate'] = df['missed_previous_appointments'] / df['previous_appointments'].replace(0,1)

In [21]:
# Encode all categorical columns
df_encoded = pd.get_dummies(df, columns=['gender','blood_group','department','diagnosis','appointment_status','room_type','payment_status','payment_method','bp_category','bmi_category'], drop_first=True)

In [22]:
# Check the new shape
df_encoded.shape

(1000, 68)

In [25]:
# Import the scaler
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [26]:
# List which columns to scale
numeric_cols = ['age','waiting_days','previous_appointments','missed_previous_appointments','length_of_stay_days','previous_admissions','systolic_bp','diastolic_bp','blood_sugar_mg_dl','cholesterol_mg_dl','bmi','lab_tests_count','treatments_count','consultation_fee_lkr','room_charge_lkr','lab_charge_lkr','medicine_charge_lkr','total_bill_lkr','missed_appointment_rate']

In [27]:
# Apply the scaling
df_encoded[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])

In [28]:
# Confirm scaling worked
df_encoded[numeric_cols].describe().T[['mean','std']]

,mean,std
age,1.421085e-17,1.0005
waiting_days,7.371881e-17,1.0005
previous_appointments,7.638334e-17,1.0005
missed_previous_appointments,-3.197442e-17,1.0005
length_of_stay_days,7.904788e-17,1.0005
previous_admissions,-3.552714e-18,1.0005
systolic_bp,-1.243450e-16,1.0005
diastolic_bp,-3.010925e-16,1.0005
blood_sugar_mg_dl,2.442491e-16,1.0005
cholesterol_mg_dl,1.909584e-16,1.0005


In [31]:
# Note on feature-target correlation
risk_map = {'Low': 0, 'Medium': 1, 'High': 2}
df_encoded['disease_risk_numeric'] = df['disease_risk_level'].map(risk_map)
correlations = df_encoded.select_dtypes(include='number').corr()['disease_risk_numeric'].sort_values(ascending=False)
correlations

disease_risk_numeric            1.000000
age                             0.537582
blood_sugar_mg_dl               0.474440
cholesterol_mg_dl               0.446766
bmi                             0.359904
systolic_bp                     0.310525
previous_admissions             0.171077
diastolic_bp                    0.084029
readmitted_30_days              0.054152
previous_appointments           0.049035
missed_previous_appointments    0.028431
waiting_days                    0.018815
medicine_charge_lkr             0.014436
room_charge_lkr                 0.013929
total_bill_lkr                  0.011739
no_show                         0.011356
record_id                       0.008862
lab_tests_count                -0.001361
treatments_count               -0.002355
admitted                       -0.002418
length_of_stay_days            -0.004472
lab_charge_lkr                 -0.006674
missed_appointment_rate        -0.008646
consultation_fee_lkr           -0.040059
Name: disease_ri

In [30]:
# Check for redundant features
df_encoded.select_dtypes(include='number').corr()

,record_id,age,waiting_days,previous_appointments,missed_previous_appointments,admitted,length_of_stay_days,previous_admissions,systolic_bp,diastolic_bp,...,lab_tests_count,treatments_count,consultation_fee_lkr,room_charge_lkr,lab_charge_lkr,medicine_charge_lkr,total_bill_lkr,no_show,readmitted_30_days,missed_appointment_rate
record_id,1.000000,-0.000308,0.007504,0.001596,0.049066,0.006652,0.008000,-0.006311,0.020529,-0.003981,...,0.080289,0.018953,-0.022413,0.017298,0.078713,-0.042321,0.022064,0.013473,0.029622,0.076265
age,-0.000308,1.000000,-0.033049,0.010143,-0.056024,-0.019014,-0.022327,-0.078275,0.221550,0.117154,...,0.014346,-0.007126,-0.005553,-0.024546,0.012843,-0.028112,-0.025917,-0.062694,0.034263,-0.066610
waiting_days,0.007504,-0.033049,1.000000,0.032579,-0.025888,0.041307,0.011456,-0.015503,0.036073,-0.079716,...,0.001851,-0.011925,0.030995,0.061850,-0.003868,-0.013425,0.046456,0.242564,0.014947,-0.006683
previous_appointments,0.001596,0.010143,0.032579,1.000000,0.373926,0.001608,0.031968,-0.010989,0.006072,0.005653,...,0.020718,-0.018000,-0.001286,0.033845,0.003509,0.038879,0.041607,-0.082226,-0.018554,-0.064792
missed_previous_appointments,0.049066,-0.056024,-0.025888,0.373926,1.000000,-0.018659,0.002899,0.015782,0.017474,0.014963,...,0.061617,-0.009458,-0.008455,0.031308,0.084157,0.031243,0.060780,0.087240,-0.052581,0.762062
admitted,0.006652,-0.019014,0.041307,0.001608,-0.018659,1.000000,0.834099,0.015205,0.030557,0.006093,...,0.408692,0.496186,0.036157,0.308706,0.353360,0.441792,0.508501,0.032540,0.829240,-0.028011
length_of_stay_days,0.008000,-0.022327,0.011456,0.031968,0.002899,0.834099,1.000000,0.018243,0.033478,-0.000015,...,0.348326,0.402739,0.012353,0.375781,0.303651,0.372148,0.523086,0.001077,0.734844,-0.033134
previous_admissions,-0.006311,-0.078275,-0.015503,-0.010989,0.015782,0.015205,0.018243,1.000000,0.026184,-0.042206,...,-0.025899,0.011592,-0.036185,0.007996,-0.024116,0.051234,0.014336,-0.010907,0.064067,0.022504
systolic_bp,0.020529,0.221550,0.036073,0.006072,0.017474,0.030557,0.033478,0.026184,1.000000,0.053947,...,0.000557,0.026534,-0.007606,0.051582,-0.003988,0.000667,0.040477,0.010675,0.026746,0.017645
diastolic_bp,-0.003981,0.117154,-0.079716,0.005653,0.014963,0.006093,-0.000015,-0.042206,0.053947,1.000000,...,-0.018439,-0.015077,-0.000639,-0.018563,-0.025250,0.025717,-0.014031,-0.042609,-0.001486,-0.023569


## Feature Engineering Process

1. Created bp_category (Normal/Elevated/High) from systolic_bp and bmi_category 
   (Underweight/Normal/Overweight/Obese) from bmi, to capture standard clinical 
   threshold effects that raw numbers alone might not represent well to a model.

2. Created missed_appointment_rate (missed_previous_appointments / 
   previous_appointments) to express missed appointments as a proportion rather 
   than a raw count, making patients with different appointment histories more 
   comparable.

3. Feature encoding: applied one-hot encoding (pd.get_dummies, drop_first=True) 
   to all categorical columns (gender, blood_group, department, diagnosis, 
   appointment_status, room_type, payment_status, payment_method, bp_category, 
   bmi_category), since ML models require numeric input. Dataset grew from 33 
   to 68 columns after encoding.

4. Feature scaling: applied StandardScaler to all numeric columns (mean 0, std 
   1 confirmed after transform), so no single feature dominates due to scale 
   differences (e.g. total_bill_lkr in thousands vs. age in tens).

5. Feature selection: checked correlation between all numeric features to 
   identify redundancy — no pairs exceeded 0.9, though room_charge_lkr/
   total_bill_lkr (0.88), lab_tests_count/lab_charge_lkr (0.876), and admitted/
   length_of_stay_days (0.834) were notably correlated, as expected given their 
   natural relationship. None were removed since none crossed the redundancy 
   threshold.

   Also checked correlation against the target (disease_risk_level, temporarily 
   mapped to 0/1/2 for this check only). Clinical features showed the strongest 
   relationship: age (0.54), blood_sugar_mg_dl (0.47), cholesterol_mg_dl (0.45), 
   bmi (0.36), and systolic_bp (0.31). Financial and operational features (e.g. 
   billing, lab test counts) showed very weak correlation (under 0.05), 
   suggesting clinical measurements will likely be the strongest predictors in 
   Task 5. This numeric mapping was for exploratory purposes only — the actual 
   target encoding for modeling will be finalized in Task 5.